In [ ]:
from mtlearn import morphology
import mtlearn

import torch
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import time

import torch
from torch.autograd import gradcheck

In [ ]:
torch.set_printoptions(precision=10, sci_mode=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float64
device

## 1 - Gradient Checks

In [ ]:
resize_proportion = 0.5

sample_image = sorted(mtlearn.data.ensure_dataset('misc256').glob('*.png'))[0]
original_image = cv.imread(str(sample_image), 0)
(num_rows, num_cols) = original_image.shape

height, width = original_image.shape
height = int(height * resize_proportion)
width = int(width * resize_proportion)

image = cv.resize(original_image, (width, height), interpolation=cv.INTER_LINEAR)

image =  np.array([ 
        [2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0],
        [2, 5, 5, 5, 3, 3, 3, 1, 1, 1, 3, 0],
        [2, 5, 6, 5, 3, 3, 3, 1, 5, 1, 3, 0],
        [2, 5, 6, 5, 3, 3, 3, 1, 6, 1, 3, 0],
        [2, 5, 6, 5, 3, 3, 3, 1, 5, 1, 3, 0],
        [2, 5, 5, 5, 3, 3, 3, 1, 1, 1, 3, 0],
        [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0],
        [2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 0],
        [6, 6, 6, 6, 6, 2, 4, 4, 4, 4, 4, 0],
        [6, 4, 6, 4, 6, 2, 4, 7, 4, 7, 4, 0],
        [6, 6, 6, 6, 6, 2, 4, 4, 4, 4, 4, 0],
        [2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0]
    ])

tree = morphology.create_max_tree(image)

plt.imshow(image, 'gray')
height, width = image.shape

### 1.1 - Attribute Extraction

In [ ]:
AttributeType = morphology.AttributeType

attributes = morphology.compute_attributes(tree, [AttributeType.AREA, AttributeType.COMPACTNESS])[1]

# Min-max normalization to the [0, 1] range
attributes = (attributes.min() - attributes) / (attributes - attributes.max() + 1e-8) 

attributes = torch.as_tensor(attributes, dtype=dtype).to(device)

### 1.2 - Parameter Initialization

In [ ]:
weight = torch.nn.Parameter(torch.empty(attributes.shape[1], dtype=dtype, device=device), requires_grad=True)
weight.data.fill_(1)

bias = torch.nn.Parameter(torch.empty(1, dtype=dtype, device=device), requires_grad=True)
bias.data.fill_(0)

beta_f = 1
clamp_logits = False

### 1.3 - Jacobian and Residue Extraction

In [ ]:
residues, pre_order, post_order, parent, node_of_pixel = mtlearn.ConnectedFilterPreprocessingTreeTensors.get_info_for_jacobian(tree)

residues = residues.to(device, dtype=dtype)
pre_order = pre_order.to(device)
post_order = post_order.to(device)
parent = parent.to(device)
node_of_pixel = node_of_pixel.to(device)

### 1.4 - Numerical Gradient Computation

Parameter initialization with positive and negative perturbations

In [ ]:
eps = 1e-08
atol = 1e-04

w0_increased = weight.clone()
w0_increased[0] += eps
w0_decreased = weight.clone()
w0_decreased[0] -= eps

w1_increased = weight.clone()
w1_increased[1] += eps
w1_decreased = weight.clone()
w1_decreased[1] -= eps

bias_increased = bias.clone()
bias_increased[0] += eps
bias_decreased = bias.clone()
bias_decreased[0] -= eps

In [ ]:
def numeric_gradient_weight(function, weight_increased, weight_decreased, bias):
    return ((function(weight_increased, bias, residues, pre_order, post_order, parent, node_of_pixel, attributes, tree.numRows, tree.numCols, beta_f) - function(weight_decreased, bias, residues, pre_order, post_order, parent, node_of_pixel, attributes, tree.numRows, tree.numCols, beta_f)) / (2 * eps)).sum()

def numeric_gradient_bias(function, weight, bias_increased, bias_decreased):
    return ((function(weight, bias_increased, residues, pre_order, post_order, parent, node_of_pixel, attributes, tree.numRows, tree.numCols, beta_f) - function(weight, bias_decreased, residues, pre_order, post_order, parent, node_of_pixel, attributes, tree.numRows, tree.numCols, beta_f)) / (2 * eps)).sum()

#### 1.4.1 - Gradient for ```weight[0]```

In [ ]:
numeric_grad_w0 = numeric_gradient_weight(
    mtlearn.layers.ConnectedFilterPreprocessingImplicitJacobianFunction.apply,
    w0_increased,
    w0_decreased,
    bias,
)

#### 1.4.2 - Gradient for ```weight[1]```

In [ ]:
numeric_grad_w1 = numeric_gradient_weight(
    mtlearn.layers.ConnectedFilterPreprocessingImplicitJacobianFunction.apply,
    w1_increased,
    w1_decreased,
    bias
)

#### 1.4.3 - Gradient for ```bias```

In [ ]:
numeric_grad_bias = numeric_gradient_bias(
    mtlearn.layers.ConnectedFilterPreprocessingImplicitJacobianFunction.apply,
    weight,
    bias_increased,
    bias_decreased
)

In [ ]:
(
    numeric_grad_w0,
    numeric_grad_w1,
    numeric_grad_bias
)

### 1.5 - Analytical Gradient

In [ ]:
filtered_image = mtlearn.layers.ConnectedFilterPreprocessingImplicitJacobianFunction.apply(weight, bias, residues, pre_order, post_order, parent, node_of_pixel, attributes, tree.numRows, tree.numCols, beta_f)

output_scalar = filtered_image.sum()
output_scalar.backward()

weight.grad, bias.grad

### 1.6 - Numerical vs Analytical Gradient Comparison

In [ ]:
(
    (weight.grad[0] - numeric_grad_w0).abs(),
    (weight.grad[1] - numeric_grad_w1).abs(),
    (bias.grad - numeric_grad_bias).abs()
)

### 1.7 - Gradient Checking

In [ ]:
success_counter = 0
fail_counter = 0

def filtered_pixel(weight, bias, pixel_index):
    return mtlearn.layers.ConnectedFilterPreprocessingImplicitJacobianFunction.apply(weight, bias, residues, pre_order, post_order, parent, node_of_pixel, attributes, tree.numRows, tree.numCols, beta_f).flatten()[pixel_index]


for pixel_index in range(len(image.flatten())):
    if pixel_index % 1000 == 0:
        print(f'{pixel_index} | Success: {success_counter} | Fail: {fail_counter}')

    parameters = (weight, bias, pixel_index)

    try:
        gradcheck(filtered_pixel, parameters, eps=eps, atol=atol)
        success_counter += 1

    except:
        fail_counter += 1

print("Success: ", success_counter)
print("Fail: ", fail_counter)